In [ ]:
# ── Validate R package installation ───────────────────────────────────────────
# UPDATE R_LIB to match your volume path
R_LIB    <- "/Volumes/<catalog>/<schema>/<volume>/r_lib"
LOCAL_SO <- "/usr/local/lib/r_syslibs"
DIAG_FILE <- "/tmp/validation_diag.txt"

diag <- function(...) {
  msg <- paste0(...)
  message(msg)
  cat(msg, "\n", file = DIAG_FILE, append = TRUE)
}
cat("", file = DIAG_FILE)

diag("R_LIB exists: ", dir.exists(R_LIB))
diag("LOCAL_SO exists: ", dir.exists(LOCAL_SO))
diag("LOCAL_SO .so count: ", length(list.files(LOCAL_SO, pattern = "\\.so")))
diag("libgdal present: ", file.exists(file.path(LOCAL_SO, "libgdal.so.34")))
diag("LD_LIBRARY_PATH: ", Sys.getenv("LD_LIBRARY_PATH"))

# syslibs: set LD_LIBRARY_PATH and dyn.load (belt + suspenders alongside ldconfig)
if (dir.exists(LOCAL_SO)) {
  Sys.setenv(LD_LIBRARY_PATH = paste(LOCAL_SO, Sys.getenv("LD_LIBRARY_PATH"), sep = ":"))
  loaded <- 0
  for (pat in c("libgeos", "libgeos_c", "libproj", "libtiff", "libjpeg", "libgdal", "libudunits2")) {
    for (f in list.files(LOCAL_SO, pattern = pat, full.names = TRUE)) {
      tryCatch({ dyn.load(f); loaded <- loaded + 1 }, error = function(e) invisible(NULL))
    }
  }
  diag("Syslibs dyn.loaded: ", loaded)
}

# r_lib first in path, unload stale versions
if (dir.exists(R_LIB)) {
  .libPaths(c(R_LIB, .libPaths()))
} else {
  stop("R_LIB not found: ", R_LIB)
}
for (pkg in c("vctrs", "rlang", "cli", "lifecycle", "glue", "magrittr")) {
  if (pkg %in% loadedNamespaces()) tryCatch(unloadNamespace(pkg), error = function(e) invisible(NULL))
}

diag("r_lib pkg count: ", length(list.files(R_LIB)))
diag("vctrs version: ", tryCatch(as.character(packageVersion("vctrs")), error=function(e) "missing"))

results <- list()
try_load <- function(pkg) {
  ok <- tryCatch({ library(pkg, character.only = TRUE); TRUE },
                 error = function(e) { diag("  [FAIL] ", pkg, ": ", conditionMessage(e)); FALSE })
  results[[pkg]] <<- ok
  ok
}

In [ ]:
# ── Core target package ───────────────────────────────────────────────────────
message("── zipcodeR ─────────────────────────────────────────")
try_load("zipcodeR")

# Spot-check: can we actually call it?
if (results[["zipcodeR"]]) {
  tryCatch({
    info <- zipcodeR::geocode_zip("90210")
    message("  geocode_zip('90210'): lat=", info$lat, " lng=", info$lng)
  }, error = function(e) message("  geocode_zip error: ", conditionMessage(e)))
}

In [ ]:
# ── Geospatial stack (requires syslibs) ───────────────────────────────────────
message("── Geospatial stack ─────────────────────────────────")
for (pkg in c("sf", "terra", "units", "s2")) try_load(pkg)

# sf: confirm GDAL/GEOS/PROJ linked correctly
if (results[["sf"]]) {
  tryCatch(sf::sf_extSoftVersion(), error = function(e) message("  sf_extSoftVersion error: ", conditionMessage(e)))
}

In [ ]:
# ── Census / tigris stack ─────────────────────────────────────────────────────
message("── Census stack ─────────────────────────────────────")
for (pkg in c("tidycensus", "tigris")) try_load(pkg)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
passed <- names(Filter(isTRUE,  results))
failed <- names(Filter(isFALSE, results))

diag(strrep("=", 50))
diag(if (length(failed) == 0) "PASS" else "FAIL",
     " — ", length(passed), "/", length(results), " packages loaded")
if (length(failed) > 0) diag("Failed: ", paste(failed, collapse = ", "))
if ("sf" %in% passed) diag("sf versions: ", paste(sf::sf_extSoftVersion(), collapse = " | "))
diag(strrep("=", 50))

# Write JSON summary to /tmp
json_out <- paste0('{"passed":', jsonlite::toJSON(passed),
                   ',"failed":', jsonlite::toJSON(failed), '}')
writeLines(json_out, "/tmp/validation_results.json")

if (length(failed) > 0) stop(paste("Validation failed:", paste(failed, collapse = ", ")))